# Gán loại tin cho dữ liệu clean

Dùng checkpoint đã chốt ở notebook 05 để dự đoán `news_type` cho phần tin chưa có nhãn. Nhãn thủ công đã xác nhận được giữ nguyên; các bài đang `review_required` không được model tự gán thay.

Notebook không train, khảo sát tham số, sửa `news_clean.csv` hoặc làm sentiment. Kết quả là một bảng theo `news_id`; điểm Softmax chỉ là điểm dự đoán, chưa phải độ tin cậy đã hiệu chỉnh.

Trước khi chạy thật, hoàn tất mục 14 của notebook 05, gồm cell lưu hồ sơ test. Notebook 06 kiểm tra đúng checkpoint, dữ liệu, tokenizer và báo cáo test; thiếu hồ sơ thì dừng trước khi dự đoán.

## 1. Thiết lập

Trên Colab, chọn GPU T4 nếu được cấp và kết nối Drive. Dùng cùng gói ZIP của dự án; checkpoint nằm trên Drive, không cần tải về máy để chạy.

In [1]:
import hashlib
import json
import os
import shutil
import subprocess
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import urlretrieve

In [2]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import torch
    from packaging.version import Version
    if not torch.cuda.is_available():
        raise RuntimeError('Chưa có CUDA. Chọn máy GPU T4 trong Colab rồi kết nối lại.')
    if Version(torch.__version__.split('+')[0]) < Version('2.6'):
        raise RuntimeError('Cần runtime Colab có PyTorch >= 2.6; không tự thay bản CUDA hiện tại.')
    print('GPU được cấp:', torch.cuda.get_device_name(0))
    print('VRAM (GB):', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))

GPU được cấp: Tesla T4
VRAM (GB): 14.6


In [3]:
if IN_COLAB:
    colab_packages = [
        'transformers==4.57.1', 'py-vncorenlp==0.1.4',
        'pandas>=2.2,<3', 'scikit-learn>=1.7,<1.8',
        'matplotlib>=3.10,<4', 'tqdm>=4.67,<5',
    ]
    subprocess.check_call([
        sys.executable, '-m', 'pip', 'install', '--quiet', *colab_packages,
    ])
    if shutil.which('java') is None:
        subprocess.check_call(['apt-get', '-qq', 'update'])
        subprocess.check_call(['apt-get', '-qq', 'install', '-y', 'openjdk-17-jre-headless'])

In [4]:
COLAB_OUTPUT_DIR = None
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    COLAB_OUTPUT_DIR = Path('/content/drive/MyDrive/stock_news_anomaly_project')

Mounted at /content/drive


In [5]:
COLAB_PROJECT_DIR = Path('/content/news_processing') if IN_COLAB else None
if IN_COLAB:
    project_archive = COLAB_OUTPUT_DIR / 'news_processing_colab.zip'
    if not project_archive.is_file():
        project_archive = Path('/content/news_processing_colab.zip')
    if project_archive.is_file():
        with zipfile.ZipFile(project_archive) as archive:
            for member in archive.infolist():
                target_path = (Path('/content') / member.filename).resolve()
                if not target_path.is_relative_to(COLAB_PROJECT_DIR.resolve()):
                    raise ValueError('Gói ZIP phải chỉ chứa thư mục news_processing, không có đường dẫn bên ngoài.')
            archive.extractall('/content')
        print('Đã cập nhật dữ liệu/code từ:', project_archive)
    elif not COLAB_PROJECT_DIR.is_dir():
        raise FileNotFoundError('Dùng Upload to Colab để tải news_processing_colab.zip vào /content.')
    print('Dữ liệu/code:', COLAB_PROJECT_DIR)
    print('Nơi lưu checkpoint/báo cáo:', COLAB_OUTPUT_DIR)

Đã cập nhật dữ liệu/code từ: /content/drive/MyDrive/stock_news_anomaly_project/news_processing_colab.zip
Dữ liệu/code: /content/news_processing
Nơi lưu checkpoint/báo cáo: /content/drive/MyDrive/stock_news_anomaly_project


In [6]:
import pandas as pd
import torch
from IPython.display import display
from tqdm.auto import tqdm
from transformers import AutoConfig, AutoModel, AutoTokenizer

In [7]:
CURRENT_DIR = Path.cwd().resolve()
PROJECT_CANDIDATES = ([COLAB_PROJECT_DIR] if IN_COLAB else []) + [
    CURRENT_DIR / 'news_processing', CURRENT_DIR, *CURRENT_DIR.parents,
]
PROJECT_DIR = next(
    (path for path in PROJECT_CANDIDATES
     if (path / 'src').is_dir() and (path / 'data').is_dir()),
    None,
)
if PROJECT_DIR is None:
    raise FileNotFoundError('Không tìm thấy news_processing chứa src/ và data/.')
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

In [8]:
from src.annotation_utils import apply_approved_label_corrections
from src.news_type_inference import (
    build_inference_plan,
    combine_news_type_predictions,
    predict_news_types,
)
from src.news_type_training import (
    load_saved_run_metadata,
    restore_saved_checkpoint,
    validate_saved_segmenter,
    verify_test_evaluation_manifest,
)
from src.phobert_cnn import PhoBertCNN, encode_news_texts

In [9]:
SAVED_RUN_NAME = '20261024_153932_380173'
BATCH_SIZE = 8
OUTPUT_ROOT = COLAB_OUTPUT_DIR if IN_COLAB else PROJECT_DIR
MODEL_ROOT = OUTPUT_ROOT / 'models/news_type_phobert_cnn'
REPORT_ROOT = OUTPUT_ROOT / 'reports/news_type'
MODEL_RUN_DIR = MODEL_ROOT / 'runs' / SAVED_RUN_NAME
SOURCE_REPORT_DIR = REPORT_ROOT / 'runs' / SAVED_RUN_NAME
VNCORENLP_DIR = PROJECT_DIR / 'models/vncorenlp'
PREDICTION_PATH = OUTPUT_ROOT / 'data/processed/news_type_predictions.csv'
PREDICTION_METADATA_PATH = PREDICTION_PATH.with_name('news_type_predictions_metadata.json')

In [10]:
INPUT_PATHS = {
    'clean': PROJECT_DIR / 'data/processed/news_clean.csv',
    'annotations': PROJECT_DIR / 'data/annotation/news_type/news_manually_labeled.csv',
    'corrections': PROJECT_DIR / 'data/annotation/news_type/label_corrections.csv',
    'excluded': PROJECT_DIR / 'data/interim/news_excluded_records.csv',
    'duplicates': PROJECT_DIR / 'data/interim/duplicate_merge_log.csv',
    'mapping': PROJECT_DIR / 'models/news_type_phobert_cnn/label_mapping.json',
    'revision': PROJECT_DIR / 'data/annotation/news_type/validation_revision.json',
    **{name: PROJECT_DIR / f'data/splits/news_type/{name}.csv'
       for name in ('train', 'validation', 'test')},
}

## 2. Phân biệt nhãn thủ công và bài cần dự đoán

Đối chiếu cả ID gốc lẫn log gộp trùng để không làm mất nhãn thủ công khi một bài được đổi sang ID chuẩn. Bài đã có quyết định chờ xem xét được giữ riêng, không coi là tin chưa từng gán nhãn.

In [11]:
missing_paths = [str(path) for path in INPUT_PATHS.values() if not path.is_file()]
if missing_paths:
    raise FileNotFoundError('Thiếu đầu vào notebook 06: ' + ', '.join(missing_paths))
source_file_hashes = {
    path.relative_to(PROJECT_DIR).as_posix(): hashlib.sha256(path.read_bytes()).hexdigest()
    for path in INPUT_PATHS.values()
}
if PREDICTION_PATH.exists() or PREDICTION_METADATA_PATH.exists():
    raise FileExistsError('Đầu ra đã tồn tại. Đối chiếu bản đã lưu trước khi chạy lại; notebook không tự ghi đè.')

In [12]:
clean_news = pd.read_csv(INPUT_PATHS['clean'], dtype=str, keep_default_na=False)
annotations = pd.read_csv(INPUT_PATHS['annotations'], dtype=str, keep_default_na=False)
approved_corrections = pd.read_csv(INPUT_PATHS['corrections'], dtype=str, keep_default_na=False)
excluded_records = pd.read_csv(INPUT_PATHS['excluded'], dtype=str, keep_default_na=False)
duplicate_merge_log = pd.read_csv(INPUT_PATHS['duplicates'], dtype=str, keep_default_na=False)
label_mapping = json.loads(INPUT_PATHS['mapping'].read_text(encoding='utf-8-sig'))
validation_revision = json.loads(INPUT_PATHS['revision'].read_text(encoding='utf-8-sig'))
label2id = label_mapping['label2id']
if sorted(label2id.values()) != list(range(len(label2id))):
    raise ValueError('Mapping nhãn không hợp lệ.')
annotations = apply_approved_label_corrections(annotations, approved_corrections)

In [13]:
inference_plan, prediction_inputs, annotation_audit = build_inference_plan(
    clean_news, annotations, excluded_records, duplicate_merge_log, label2id,
)
plan_summary = (
    inference_plan['label_source'].value_counts()
    .rename_axis('label_source').reset_index(name='article_count')
)
display(plan_summary)
print('Tổng bài clean:', len(clean_news))
print('Bài cần model dự đoán:', len(prediction_inputs))

,label_source,article_count
0,unlabeled,2260
1,manual,1058
2,review_required,38


Tổng bài clean: 3356
Bài cần model dự đoán: 2260


`manual` là nhãn đã chốt; `unlabeled` là phần sẽ được model xử lý; `review_required` giữ trống nhãn đầu ra để tránh thay thế quyết định chưa chốt của người gán. Đây là thống kê luồng xử lý, không phải chỉ số chất lượng model.

## 3. Kiểm tra model và điều kiện bàn giao

Chọn đúng lần chạy, không tự tìm checkpoint mới nhất. Giữ nguyên quy trình mã hóa và kiến trúc trong cấu hình đã lưu. Báo cáo test phải thuộc chính model và đầu vào hiện tại; notebook không tự chạy test hoặc áp ngưỡng điểm để chọn lại model.

In [14]:
if (not SAVED_RUN_NAME or SAVED_RUN_NAME in {'.', '..'}
        or '/' in SAVED_RUN_NAME or '\\' in SAVED_RUN_NAME):
    raise ValueError('SAVED_RUN_NAME phải là tên một lần chạy, không phải đường dẫn.')
if MODEL_RUN_DIR.resolve().parent != (MODEL_ROOT / 'runs').resolve():
    raise ValueError('Checkpoint phải nằm trong thư mục runs đã chỉ định.')
config_path = MODEL_RUN_DIR / 'run_config.json'
if not config_path.is_file():
    raise FileNotFoundError('Thiếu model đã lưu. Chạy trên Colab có Drive hoặc đồng bộ đủ model về máy.')
saved_config_preview = json.loads(config_path.read_text(encoding='utf-8-sig'))

In [15]:
input_file_hashes = {
    INPUT_PATHS[name].relative_to(PROJECT_DIR).as_posix(): source_file_hashes[
        INPUT_PATHS[name].relative_to(PROJECT_DIR).as_posix()
    ] for name in ('train', 'validation', 'test', 'mapping')
}
active_revision = (
    validation_revision if saved_config_preview['input_sha256'] != input_file_hashes else None
)
run_config, run_history = load_saved_run_metadata(
    MODEL_RUN_DIR, SOURCE_REPORT_DIR, input_sha256=input_file_hashes,
    label_mapping=label_mapping,
    expected_config={'model_name': 'vinai/phobert-base-v2', 'num_classes': len(label2id),
                     'word_segmenter': 'VnCoreNLP wseg 1.2',
                     'pooling_special_tokens': 'include; padding excluded'},
    validation_revision=active_revision, validation_path=INPUT_PATHS['validation'],
)

In [16]:
evaluation_revision = run_history.attrs.get('evaluation_revision')
TEST_REPORT_DIR = (
    REPORT_ROOT / 'evaluations' / evaluation_revision if evaluation_revision else SOURCE_REPORT_DIR
)
revision_sha256 = (
    hashlib.sha256(INPUT_PATHS['revision'].read_bytes()).hexdigest() if active_revision else None
)
test_data = pd.read_csv(INPUT_PATHS['test'], dtype=str, keep_default_na=False)
MAX_LENGTH = run_config['max_length']
HEAD_RATIO = run_config['head_ratio']

In [17]:
test_verification_arguments = {
    'input_sha256': input_file_hashes, 'label_mapping': label_mapping,
    'validation_revision_sha256': revision_sha256,
    'segmenter_sha256': run_config['segmenter_sha256'],
    'max_length': MAX_LENGTH, 'head_ratio': HEAD_RATIO, 'test_data': test_data,
}
test_handoff = verify_test_evaluation_manifest(
    MODEL_RUN_DIR, TEST_REPORT_DIR, **test_verification_arguments,
)
print('Đã đối chiếu hồ sơ test của:', test_handoff['source_run'])

Đã đối chiếu hồ sơ test của: 20261024_153932_380173


## 4. Tách từ và mã hóa phần chưa có nhãn

Chỉ mã hóa những bài cần dự đoán. VnCoreNLP, tokenizer, giới hạn token và tỷ lệ Head–Tail phải trùng model đã lưu. Gắn `news_id` vào từng bản mã hóa để kiểm tra thứ tự khi ghép kết quả.

In [18]:
java_executable_name = 'java.exe' if os.name == 'nt' else 'java'
java_home_candidates = [
    Path(os.environ[name]) for name in ('JAVA_HOME', 'JDK_HOME', 'JRE_HOME')
    if os.environ.get(name)
]
java_home_candidates.extend(sorted((PROJECT_DIR / 'models/java').glob('*')))
java_on_path = shutil.which('java')
if java_on_path:
    java_home_candidates.append(Path(java_on_path).resolve().parent.parent)
java_home = next(
    (path.resolve() for path in java_home_candidates
     if (path / 'bin' / java_executable_name).is_file()),
    None,
)
if java_home is not None:
    os.environ['JAVA_HOME'] = str(java_home)
    java_bin_path = str(java_home / 'bin')
    if java_bin_path not in os.environ.get('PATH', '').split(os.pathsep):
        os.environ['PATH'] = java_bin_path + os.pathsep + os.environ.get('PATH', '')

In [19]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if shutil.which('java') is None:
    raise RuntimeError('Cần Java để chạy VnCoreNLP trước khi mã hóa.')
print('Thiết bị:', DEVICE)
if DEVICE.type == 'cpu' and len(prediction_inputs):
    print('Đang dùng CPU; suy luận PhoBERT có thể chậm. Nên dùng Colab T4.')

Thiết bị: cuda


In [20]:
vncorenlp_base_url = 'https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/'
vncorenlp_resources = (
    'VnCoreNLP-1.2.jar',
    'models/wordsegmenter/vi-vocab',
    'models/wordsegmenter/wordsegmenter.rdr',
)
for relative_path in vncorenlp_resources:
    resource_path = VNCORENLP_DIR / relative_path
    if resource_path.is_file() and resource_path.stat().st_size > 0:
        continue
    resource_path.parent.mkdir(parents=True, exist_ok=True)
    download_path = resource_path.with_name(resource_path.name + '.part')
    urlretrieve(vncorenlp_base_url + relative_path, str(download_path))
    if download_path.stat().st_size == 0:
        raise RuntimeError(f'Tải tài nguyên không thành công: {relative_path}')
    download_path.replace(resource_path)

In [21]:
segmenter_file_hashes = {
    relative_path: hashlib.sha256((VNCORENLP_DIR / relative_path).read_bytes()).hexdigest()
    for relative_path in vncorenlp_resources
}
validate_saved_segmenter(run_config, segmenter_file_hashes)

In [22]:
import py_vncorenlp

working_directory = Path.cwd()
try:
    segmenter = py_vncorenlp.VnCoreNLP(annotators=['wseg'], save_dir=str(VNCORENLP_DIR))
finally:
    # Wrapper đổi working directory khi khởi tạo; trả lại thư mục notebook.
    os.chdir(working_directory)

In [23]:
tokenizer = AutoTokenizer.from_pretrained(
    str(MODEL_RUN_DIR / 'tokenizer'), use_fast=False, local_files_only=True,
)
print('Ngân sách token nội dung:', MAX_LENGTH - tokenizer.num_special_tokens_to_add(pair=False))

Ngân sách token nội dung: 254


In [24]:
prediction_encodings = []
if not prediction_inputs.empty:
    prediction_encodings = encode_news_texts(
        tqdm(prediction_inputs['input_text'], desc='Mã hóa bài chưa có nhãn'),
        segmenter, tokenizer, max_length=MAX_LENGTH, head_ratio=HEAD_RATIO,
    )
    for news_id, encoding in zip(prediction_inputs['news_id'], prediction_encodings):
        encoding['news_id'] = news_id
print('Bài đã mã hóa:', len(prediction_encodings))

Mã hóa bài chưa có nhãn:   0%|          | 0/2260 [00:00<?, ?it/s]

Bài đã mã hóa: 2260


## 5. Dự đoán loại tin

Khôi phục toàn bộ trọng số đã fine-tune, chuyển model sang `eval()` và suy luận theo batch. Không tạo optimizer, hàm loss hoặc nhãn giả; chỉ đọc điểm đầu ra để chọn loại tin.

In [25]:
backbone_config = AutoConfig.from_pretrained(
    str(MODEL_RUN_DIR / 'backbone'), local_files_only=True,
)
backbone = AutoModel.from_config(backbone_config)
model = PhoBertCNN(
    backbone=backbone, num_classes=run_config['num_classes'],
    kernel_sizes=run_config['kernel_sizes'], num_filters=run_config['num_filters'],
    global_projection_size=run_config['global_projection_size'],
    dropout=run_config['dropout'],
).to(DEVICE)
checkpoint_metadata = restore_saved_checkpoint(model, MODEL_RUN_DIR, run_config, DEVICE)
print('Đã nạp checkpoint epoch:', checkpoint_metadata['epoch'])

Đã nạp checkpoint epoch: 2


In [26]:
model_predictions = predict_news_types(
    model, prediction_inputs, prediction_encodings, DEVICE, label2id, batch_size=BATCH_SIZE,
)
print('Bài được model dự đoán:', len(model_predictions))

Bài được model dự đoán: 2260


## 6. Ghép kết quả và kiểm tra độ bao phủ

Mỗi bài clean xuất hiện đúng một lần, giữ nguyên thứ tự. Nhãn thủ công không bị model ghi đè; bài cần rà lại vẫn để trống `news_type`. Không gọi số lượng nhãn dự đoán là Accuracy hoặc F1 vì phần này không có nhãn chuẩn để tính.

In [27]:
news_type_predictions = combine_news_type_predictions(
    inference_plan, model_predictions, label2id,
)
if news_type_predictions['news_id'].tolist() != clean_news['news_id'].tolist():
    raise AssertionError('Kết quả không giữ đúng danh sách và thứ tự bài clean.')
display(news_type_predictions.head(10))

,news_id,news_type,label_source,prediction_score
0,57a3bac8e0b55bd495bf688a17e6a9f3a3b6fef06373ae...,business_operations,model,0.994331
1,9cddfe019c634b81361fb00a77a9bae5bbc61795e00ce8...,business_operations,model,0.993245
2,e1751ac1f9ba5c069ce22bae353e052684d1f59c8d23b3...,business_operations,model,0.998573
3,f97a7efa668f1e2d51930ff67d119c64177b3209afd7e6...,business_operations,model,0.998415
4,bfc140797b8dca4f01f53cc8125ed58a1c172ef49303b3...,financial_results,model,0.995802
5,69015caad83d18da77c124e9b6823306044480f3705df0...,financial_results,manual,
6,832fa55a92cc7ba6c4369400ee0f08cc71e3e99a479f7d...,financial_results,model,0.993025
7,a6cac2bb5914a44fa39a83bcab1aefed051ebe04004cb1...,business_operations,model,0.998901
8,acc8bf749113715eaf5b1f4e4a819257eefc1273da81f1...,business_operations,manual,
9,a91599a1475fca3bbcc23bf3f446e139e812e2c0ad838a...,financial_results,model,0.988516


In [28]:
prediction_summary = (
    news_type_predictions.groupby(['label_source', 'news_type'], dropna=False)
    .size().rename('article_count').reset_index()
)
display(prediction_summary)
print('Bài xuất kết quả:', len(news_type_predictions))
print('Bài chưa có nhãn chốt:', int(news_type_predictions['label_source'].eq('review_required').sum()))

,label_source,news_type,article_count
0,manual,business_operations,302
1,manual,financial_results,176
2,manual,governance_capital,153
3,manual,industry,100
4,manual,legal_compliance,50
5,manual,mention_only,77
6,manual,stock_market,200
7,model,business_operations,934
8,model,financial_results,455
9,model,governance_capital,499


Bài xuất kết quả: 3356
Bài chưa có nhãn chốt: 38


## 7. Lưu kết quả

Xuất một CSV gồm `news_id`, `news_type`, `label_source`, `prediction_score` và một JSON lưu nguồn model/dữ liệu. Nhãn thủ công và bài chờ không có điểm model. Hai file được lưu ở `data/processed` trên Drive khi chạy Colab; `news_clean.csv` và annotation gốc không đổi.

In [29]:
for path in INPUT_PATHS.values():
    relative_path = path.relative_to(PROJECT_DIR).as_posix()
    if hashlib.sha256(path.read_bytes()).hexdigest() != source_file_hashes[relative_path]:
        raise RuntimeError('Đầu vào thay đổi trong lúc dự đoán: ' + relative_path)
if verify_test_evaluation_manifest(
    MODEL_RUN_DIR, TEST_REPORT_DIR, **test_verification_arguments,
) != test_handoff:
    raise RuntimeError('Model hoặc hồ sơ test thay đổi trong lúc dự đoán.')
if PREDICTION_PATH.exists() or PREDICTION_METADATA_PATH.exists():
    raise FileExistsError('Không ghi đè kết quả đã có.')

In [30]:
prediction_csv_bytes = news_type_predictions.to_csv(
    index=False, lineterminator='\n',
).encode('utf-8-sig')
prediction_csv_sha256 = hashlib.sha256(prediction_csv_bytes).hexdigest()
label_source_counts = news_type_predictions['label_source'].value_counts().to_dict()

In [31]:
prediction_metadata = {
    'schema_version': 1, 'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'source_run': SAVED_RUN_NAME, 'checkpoint_epoch': checkpoint_metadata['epoch'],
    'checkpoint_sha256': test_handoff['checkpoint_sha256'],
    'test_manifest_sha256': hashlib.sha256(
        (TEST_REPORT_DIR / 'test_evaluation_config.json').read_bytes()
    ).hexdigest(),
    'input_file_sha256': source_file_hashes, 'label_mapping': label_mapping,
    'preprocessing': {'max_length': MAX_LENGTH, 'head_ratio': HEAD_RATIO,
                      'segmenter_sha256': segmenter_file_hashes},
    'label_source_counts': label_source_counts,
    'output_rows': len(news_type_predictions), 'output_columns': news_type_predictions.columns.tolist(),
    'output_csv_sha256': prediction_csv_sha256, 'human_annotations_modified': False,
    'score_description': 'Uncalibrated maximum Softmax score; blank for manual/review_required.',
}
prediction_metadata_bytes = (
    json.dumps(prediction_metadata, ensure_ascii=False, indent=2) + '\n'
).encode('utf-8')

In [32]:
created_paths = []
try:
    PREDICTION_PATH.parent.mkdir(parents=True, exist_ok=True)
    for output_path, payload in (
        (PREDICTION_PATH, prediction_csv_bytes),
        (PREDICTION_METADATA_PATH, prediction_metadata_bytes),
    ):
        with output_path.open('xb') as output_file:
            created_paths.append(output_path)
            output_file.write(payload)
    if hashlib.sha256(PREDICTION_PATH.read_bytes()).hexdigest() != prediction_csv_sha256:
        raise RuntimeError('File kết quả không khớp dữ liệu vừa dự đoán.')
except BaseException:
    for created_path in reversed(created_paths):
        created_path.unlink(missing_ok=True)
    raise
print('Kết quả:', PREDICTION_PATH)
print('Nguồn model và dữ liệu:', PREDICTION_METADATA_PATH)

Kết quả: /content/drive/MyDrive/stock_news_anomaly_project/data/processed/news_type_predictions.csv
Nguồn model và dữ liệu: /content/drive/MyDrive/stock_news_anomaly_project/data/processed/news_type_predictions_metadata.json
